# 04a — Logistic Regression: Environmental Analysis

Models MRSA acquisition as a function of ward-level colonization pressure, in the
**environmental** cohort. This cohort is matched on age, sex, prior surgery, room LOS, and
antibiotic exposure history — so age/sex are **not** added as covariates here (matching
already balances them; see the check in `01_eda.ipynb`).

Primary predictor: `MRSA_cp`. Other pathogens' CP columns and the Elixhauser index are
included as covariates to see whether MRSA-specific colonization pressure holds up once
general ward "dirtiness" and comorbidity burden are accounted for.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor

sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_processed

env = load_processed("environmental_mrsa.csv")
env.shape

(3498, 79)

## Build design matrix

In [2]:
cp_cols = [c for c in env.columns if c.endswith("_cp")]
elix_cols = [c for c in env.columns if c.startswith("elix_") and c != "elix_index_mortality"]

predictors = cp_cols + ["any_surgery", "elix_index_mortality"]
model_df = env[["group_binary"] + predictors].dropna()
print(model_df.shape)
model_df.describe()

(3498, 12)


,group_binary,DS_Entero_cp,ESBL_cp,CDiff_cp,VSE_cp,VRE_cp,MSSA_cp,MRSA_cp,DS_PsA_cp,DR_PsA_cp,any_surgery,elix_index_mortality
count,3498.000000,3498.000000,3498.000000,3498.000000,3498.000000,3498.000000,3498.000000,3498.000000,3498.000000,3498.000000,3498.000000,3498.000000
mean,0.314751,12.155734,3.501333,1.134946,3.209614,1.180502,3.769285,2.000656,2.131157,0.974428,0.114351,8.377644
std,0.464483,9.235874,3.167528,1.443969,2.998795,1.494042,2.829423,1.827047,2.031367,1.221290,0.318283,15.267476
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-33.000000
25%,0.000000,4.715808,0.911341,0.000000,0.785606,0.000000,1.590435,0.683281,0.000000,0.000000,0.000000,0.000000
50%,0.000000,10.538168,2.930353,0.756382,2.502656,0.778501,3.352387,1.620225,1.725228,0.730370,0.000000,4.000000
75%,1.000000,17.863769,5.213363,1.693215,4.905506,1.756162,5.441653,3.083642,3.361363,1.592446,0.000000,18.000000
max,1.000000,54.660351,29.003573,9.740136,20.004883,11.116431,18.750516,11.440696,12.880266,8.514681,1.000000,90.000000


## Multicollinearity check (VIF)

In [3]:
X = sm.add_constant(model_df[predictors])
vif = pd.DataFrame({
    "variable": X.columns,
    "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
})
vif

,variable,VIF
0,const,3.986014
1,DS_Entero_cp,2.717546
2,ESBL_cp,2.060114
3,CDiff_cp,1.204791
4,VSE_cp,2.062365
5,VRE_cp,1.550920
6,MSSA_cp,1.447117
7,MRSA_cp,1.683553
8,DS_PsA_cp,1.905794
9,DR_PsA_cp,1.443417


## Fit logistic regression

In [4]:
X = sm.add_constant(model_df[predictors])
y = model_df["group_binary"]

logit_env = sm.Logit(y, X).fit()
logit_env.summary()

Optimization terminated successfully.
         Current function value: 0.618477
         Iterations 5


<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:           group_binary   No. Observations:                 3498
Model:                          Logit   Df Residuals:                     3486
Method:                           MLE   Df Model:                           11
Date:                Mon, 17 Aug 2026   Pseudo R-squ.:                0.007021
Time:                        13:32:41   Log-Likelihood:                -2163.4
converged:                       True   LL-Null:                       -2178.7
Covariance Type:            nonrobust   LLR p-value:                  0.001278
========================================================================================
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -0.6732      0.072     -9.298      0.000      -0.815      -0.531
DS_Entero_cp            -0.0193      0.007     -2.885      0.004      -0.032      -0.006
ESBL_cp                 -0.0253      0.017     -1.493      0.135      -0.059       0.008
CDiff_cp                -0.0107      0.028     -0.377      0.706      -0.066       0.045
VSE_cp                  -0.0133      0.018     -0.742      0.458      -0.048       0.022
VRE_cp                   0.0470      0.031      1.542      0.123      -0.013       0.107
MSSA_cp                 -0.0055      0.016     -0.351      0.726      -0.036       0.025
MRSA_cp                  0.0665      0.026      2.571      0.010       0.016       0.117
DS_PsA_cp                0.0202      0.025      0.805      0.421      -0.029       0.069
DR_PsA_cp                0.0202      0.036      0.562      0.574      -0.050       0.091
any_surgery             -0.0425      0.116     -0.368      0.713      -0.269       0.184
elix_index_mortality     0.0047      0.002      1.970      0.049    2.43e-05       0.009
========================================================================================
"""

## Odds ratios with 95% CIs

In [5]:
params = logit_env.params
conf = logit_env.conf_int()
conf.columns = ["ci_low", "ci_high"]
or_table = np.exp(pd.concat([params, conf], axis=1).rename(columns={0: "coef"}))
or_table.columns = ["OR", "ci_low", "ci_high"]
or_table.sort_values("OR", ascending=False)

,OR,ci_low,ci_high
MRSA_cp,1.068786,1.015935,1.124385
VRE_cp,1.048148,0.987325,1.112718
DR_PsA_cp,1.020443,0.950961,1.095001
DS_PsA_cp,1.020391,0.971450,1.071797
elix_index_mortality,1.004733,1.000024,1.009464
MSSA_cp,0.994564,0.964725,1.025326
CDiff_cp,0.989352,0.935857,1.045905
VSE_cp,0.986801,0.952791,1.022026
DS_Entero_cp,0.980872,0.968086,0.993827
ESBL_cp,0.975001,0.943125,1.007955


## Interpretation

Focus on `MRSA_cp`'s odds ratio and CI: does ward-level MRSA colonization pressure predict
acquisition after controlling for general ward colonization pressure (other CP columns),
prior surgery, and comorbidity burden?

In [6]:
import pickle
Path("../reports").mkdir(exist_ok=True)
with open("../reports/logit_environmental.pkl", "wb") as f:
    pickle.dump(logit_env, f)